# Momentum & Nesterov — On the importance of initialization and momentum in deep learning (2013)

_Papers · Foundations & Optimization_

**A velocity that accumulates across steps lets gradient descent rush down long shallow valleys and damp the side-to-side bouncing — and Nesterov's lookahead makes it more stable still.**

---

This notebook is a practice scaffold for the **Momentum & Nesterov — On the importance of initialization and momentum in deep learning (2013)** lesson. We build classical momentum and Nesterov's lookahead from scratch one step at a time, check them against PyTorch's optimizer, then watch both race down a narrow ravine. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Set up the quadratic objective and the update rule

We optimize a simple quadratic $f(\theta)=\tfrac12\,\theta^\top A\theta$, whose gradient is just $A\theta$. A quadratic is the cleanest place to see momentum work because its curvature is fixed and easy to reason about.

We also fix the momentum coefficient $\mu$, the learning rate, and the number of steps once, so every method below shares identical hyperparameters.

In [ ]:
import torch

torch.manual_seed(0)

# A simple quadratic objective f(theta) = 0.5 * theta^T A theta  ->  grad = A @ theta
A = torch.tensor([[3.0, 0.2],
                  [0.2, 1.0]])
theta0 = torch.tensor([1.5, -2.0])

mu = 0.9       # momentum coefficient: how much velocity carries over each step
lr = 0.1       # learning rate (the paper's epsilon)
steps = 6      # how many update steps to run

def grad(theta):
    return A @ theta

### Step 2 — Build classical momentum and Nesterov from scratch

**Classical momentum** (paper eqs. 1–2) decays the velocity, subtracts `lr * gradient` evaluated *at the current* $\theta$, then moves by that velocity.

**Nesterov (NAG)** (eqs. 3–4) differs in one place: it first peeks ahead to $\theta + \mu v$ and takes the gradient *there*, so it reacts to where the velocity is about to carry it rather than where it currently sits.

In [ ]:
# ---- Classical momentum from scratch: v = mu*v - lr*g ; theta += v  (paper eqs. 1-2) ----
@torch.no_grad()
def my_cm():
    theta = theta0.clone()
    v = torch.zeros_like(theta)
    for _ in range(steps):
        g = grad(theta)            # gradient AT the current theta
        v = mu*v - lr*g            # eq. (1): decay velocity, subtract lr*gradient
        theta = theta + v          # eq. (2): move by velocity
    return theta

# ---- Nesterov from scratch: gradient at the LOOKAHEAD theta + mu*v  (paper eqs. 3-4) ----
@torch.no_grad()
def my_nag():
    theta = theta0.clone()
    v = torch.zeros_like(theta)
    for _ in range(steps):
        g = grad(theta + mu*v)     # eq. (3): gradient at the lookahead point
        v = mu*v - lr*g
        theta = theta + v          # eq. (4)
    return theta

### Step 3 — Check it against PyTorch's optimizer

Our hand-written classical momentum must match `torch.optim.SGD(momentum=mu)` exactly — that is the oracle that proves our code is correct. We feed the optimizer the *same* gradient $A\theta$ each step and compare the final $\theta$.

NAG is also compared, but note: PyTorch folds the lookahead into its momentum buffer, an equivalent but reparameterized update, so its NAG iterates are not step-for-step equal to our lookahead form — only classical momentum is the clean oracle.

In [ ]:
# ---- THE ORACLE: classical momentum must equal torch.optim.SGD(momentum=mu) ----
p = torch.nn.Parameter(theta0.clone())
opt = torch.optim.SGD([p], lr=lr, momentum=mu)
for _ in range(steps):
    opt.zero_grad()
    p.grad = (A @ p.detach()).clone()   # same gradient A@theta
    opt.step()

mine = my_cm()
print("classical momentum, my theta:  ", mine.tolist())
print("torch.optim.SGD(momentum) theta:", p.detach().tolist())
print("allclose(CM, SGD) after", steps, "steps:",
      torch.allclose(mine, p.detach(), atol=1e-6))          # expect True
print("max abs diff:", (mine - p.detach()).abs().max().item())

# ---- NAG: paper's lookahead form vs PyTorch's nesterov buffer form (equivalent reparam.) ----
pn = torch.nn.Parameter(theta0.clone())
optn = torch.optim.SGD([pn], lr=lr, momentum=mu, nesterov=True)
for _ in range(steps):
    optn.zero_grad()
    pn.grad = (A @ pn.detach()).clone()
    optn.step()
print("\nNAG my (lookahead) theta:", my_nag().tolist())
print("NAG torch nesterov theta:", pn.detach().tolist())
print("NOTE: these differ — PyTorch folds the lookahead into its momentum buffer (a different but",
      "equivalent parameterization), so the iterates are not step-for-step equal. CM is the clean oracle.")

### Step 4 — Reproduce the worked-example trajectory by hand

Finally we replay the lesson's scalar worked example: $f(x)=\tfrac12 x^2$ so the gradient is just $x$, with $lr=0.1$, $\mu=0.9$, starting at $x_0=1$ and $v_0=0$.

Printing $g$, $v$, and $x$ at each step lets you trace the velocity accumulating and confirm the expected values $x = 0.9000,\ 0.7200,\ 0.4860,\ 0.2268$.

In [ ]:
# ---- recompute the worked example: f(x)=0.5 x^2, grad=x, lr=0.1, mu=0.9, x0=1, v0=0 ----
x = 1.0
v = 0.0
for t in range(1, 5):
    g = x                  # gradient of 0.5 x^2 is x
    v = 0.9*v - 0.1*g      # classical-momentum velocity update
    x = x + v              # move by velocity
    print(f"worked step {t}: g={g:.4f}  v={v:.4f}  x={x:.4f}")
# expect x: 0.9000, 0.7200, 0.4860, 0.2268

## Visualize the data & results

_Minimize the same long, narrow ravine loss from the same start with plain SGD (mu=0), classical momentum (mu=0.9), and NAG (mu=0.9) at the same learning rate — does momentum accelerate down the shallow floor, and does NAG damp the side-to-side bouncing across the steep walls?_

### Step 1 — Define the ravine and the shared optimizer loop

Here the loss is a long, narrow **ravine**: steep across $x$ (the walls, coefficient `a=8`) and shallow along $y$ (the floor, coefficient `b=0.4`). This geometry is exactly where momentum shines — and where plain SGD struggles.

We write one `run` function that does classical momentum, and flips to the Nesterov lookahead gradient when `nesterov=True`, so all three methods share identical code and start point.

In [ ]:
import numpy as np

# ravine: steep across x (walls), shallow along y (floor)
a = 8.0
b = 0.4

def f(p):
    return 0.5*(a*p[0]**2 + b*p[1]**2)

def grad(p):
    return np.array([a*p[0], b*p[1]])

lr = 0.05
mu = 0.9
steps = 60
start = np.array([1.0, 6.0])          # near the wall on x, far out on the shallow floor y

def run(mu, nesterov=False):
    p = start.copy()
    v = np.zeros(2)
    loss = [f(p)]
    xs = [p[0]]
    for _ in range(steps):
        g = grad(p + mu*v) if nesterov else grad(p)   # NAG: gradient at the lookahead
        v = mu*v - lr*g                                # v = mu*v - lr*g
        p = p + v                                      # theta += v
        loss.append(f(p))
        xs.append(p[0])
    return np.array(loss), np.array(xs)

### Step 2 — Run all three methods and compare

We run plain SGD ($\mu=0$), classical momentum ($\mu=0.9$), and NAG ($\mu=0.9$) from the same start at the same learning rate.

The two printed summaries answer the two questions: the final loss shows whether momentum accelerates down the shallow floor, and the *wall swing* (max $|x|$ after warm-up) shows whether NAG damps the side-to-side bouncing across the steep walls.

In [ ]:
l_sgd, _      = run(0.0)              # plain SGD (mu = 0)
l_cm,  x_cm   = run(mu)               # classical momentum
l_nag, x_nag  = run(mu, nesterov=True)

print("final loss  -> SGD: %.4f  CM: %.4f  NAG: %.4f" % (l_sgd[-1], l_cm[-1], l_nag[-1]))
print("wall swing (max|x| after warm-up) -> CM: %.3f  NAG: %.3f" %
      (np.max(np.abs(x_cm[3:])), np.max(np.abs(x_nag[3:]))))
# final loss -> SGD: 0.6375  CM: 0.0019  NAG: 0.0004
# wall swing -> CM: 0.810  NAG: 0.260

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Do one classical-momentum step on $f(x)=\tfrac12 x^2$ ($\nabla f=x$) from $x_0=2$ with a velocity already at $v_0=-0.5$, using $\varepsilon=0.1$, $\mu=0.9$. Then say what plain SGD would have done from the same point.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Gradient at the current point: $g=\nabla f(2)=2$. — _CM uses the gradient at $\theta_t$._
- Velocity: $v_1=0.9\cdot(-0.5)-0.1\cdot 2=-0.45-0.2=-0.65$. — _Decay old velocity, add the fresh step._
- Update: $x_1=2+(-0.65)=1.35$. — _Move by the new velocity._
- Plain SGD ($\mu=0$, no stored velocity) would move $x=2-0.1\cdot 2=1.8$. — _No accumulation._

**Answer:** CM moves to $x_1=1.35$; plain SGD only reaches $1.8$. The carried-over velocity ($-0.5$) plus the fresh gradient step makes CM travel $0.65$ this step versus SGD's $0.2$. That extra distance is the accumulated velocity at work — exactly why momentum is faster along a consistent descent direction.

</details>

**Problem 2.** Ablation: $\mu=0$ vs $\mu=0.9$. On the same ravine loss, why does $\mu=0.9$ reach a far lower loss than $\mu=0$ in the same number of steps — and what is the risk of pushing $\mu$ even higher with the same learning rate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set $\mu=0$: the velocity is just $-\varepsilon g$, i.e. plain SGD. It must keep $\varepsilon$ tiny to not overshoot the steep walls, so it crawls along the shallow floor. — _One small step size has to serve both directions._
- Set $\mu=0.9$: along the floor the same-signed gradients accumulate, amplifying the effective step by about $\frac{1}{1-0.9}=10\times$; across the walls the sign-flipping gradients cancel. — _Velocity amplifies persistent descent, damps oscillation._
- Push $\mu\to0.99$ at the same $\varepsilon$: the effective step balloons (~$100\times$) and can overshoot the walls. — _Momentum and learning rate interact._

**Answer:** With $\mu=0.9$ the velocity accumulates down the long shallow floor where plain SGD ($\mu=0$) crawls, so momentum reaches a much lower loss in the same steps. In our CODEVIZ run, after 60 steps classical momentum reaches loss ~0.0019 versus plain SGD's ~0.637 — over 300× lower. But too-large $\mu$ at the same learning rate overshoots the steep walls and can diverge, which is why the paper schedules $\mu$ up slowly rather than just maxing it out.

</details>

**Problem 3.** Why does NAG bounce across the ravine walls less than classical momentum at the same $\mu$ and $\varepsilon$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- CM computes the gradient at the current point $\theta_t$, before the velocity has moved it. — _It reacts to where you are._
- NAG first applies the decayed velocity to reach the lookahead $\theta_t+\mu v_t$, then takes the gradient there. — _It reacts to where you are about to be._
- If $\mu v_t$ is about to overshoot a wall, the gradient at the lookahead points back toward $\theta_t$ more strongly, correcting $v$ sooner. — _A larger and more timely correction (Section 2.1)._

**Answer:** NAG's lookahead lets it see the bad overshoot one step early and pull the velocity back before it happens, while CM only notices after stepping. The paper (Section 2.1, Figure 1) shows this makes NAG "more tolerant of large values of $\mu$" with smaller oscillations. In our CODEVIZ run the wall-direction swing shrinks from ~0.81 (CM) to ~0.26 (NAG), close to plain SGD's ~0.22, while NAG still reaches the lowest loss of the three.

</details>